# Analysis

## Setting parameters

In [55]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [110]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass',   
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # todo temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
    # 'ecoli',
    # 'wisconsin',
    # 'dermatology',
    # 'cleveland',
    # 'spectfheart',
    # 'bupa',
    # 'yeast',
    # 'heart',
    # 'australian',
    # 'glass',
    # 'pima',
    'australian',
     'bands',
     'bupa',
     'cleveland',
     'dermatology',
     'ecoli',
     'glass',   
     'heart',
     'ionosphere',
     'pima',
     'sonar',  # 70 features! eventjes skip voor QuickReduct 25/04/2024
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

qr_list = [
    'australian',
    'bupa',
    'cleveland',
    'dermatology',
    'ecoli',
    'glass',
    'heart',
    'pima',
    'spectfheart',
    # 'vehicle',
    # 'vowel',
    'wisconsin',
    'yeast'
]

In [111]:
data_sets = inclusion_test

In [112]:
len(data_sets)

16

In [113]:
data_base = Path.cwd() / 'keel-data'
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [114]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [115]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [116]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [117]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )

## Adding results for non rule based models


In [118]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [119]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [120]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in data_sets}

## FRRI Models

In [249]:
frri_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    # 'lower-new-check' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / results_base / 'lower-approx-luka-impl-incl.99'
    'incl 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "1-1e-6",
    'incl 1e-4' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    'incl 1e-3' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "999",
    'incl 1e-2' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "99",
    # 'incl 0.95' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "95",
    'owa 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'linear-owa-inclusion' / '1-1e-6',
    # 'quickreduct 1e-6': Path.cwd() / results_base / 'lower-approx' / 'quickreduct-ordering-bug' / '1-1e-6',
    'cv-thres': Path.cwd() / results_base / 'lower-approx' / 'cv-inclusion',
    'frfs': Path.cwd() / results_base / 'lower-approx' / 'frfs',
    'msecvx-no-reducts': Path.cwd() / results_base / 'multiclassMSECVX' / 'no-reducts',
    'maecvx-no-reducts': Path.cwd() / results_base / 'multiclassMAECVX' / 'no-reducts',
    'base-no-reducts': Path.cwd() / results_base / 'inclusion' / 'no-reducts',
}

In [250]:
for model, path in frri_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

incl 1e-6
incl 1e-4
incl 1e-3
incl 1e-2
owa 1e-6
cv-thres
frfs
msecvx-no-reducts
maecvx-no-reducts
base-no-reducts


## Checking all of the models

In [251]:
scores.keys()

dict_keys(['modlem', 'furia', 'ripper', 'qr', 'naive-bayes', 'svm', 'tree', 'frnn', 'incl 1e-6', 'incl 1e-4', 'incl 1e-3', 'incl 1e-2', 'owa 1e-6', 'cv-thres', 'frfs', 'msecvx-no-reducts', 'maecvx-no-reducts', 'base-no-reducts'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [252]:
names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    'frri-paper': [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
        'furia',
        'ripper',
    ],
    'inclusion' : [
        'incl 1e-6',
        # 'incl 1e-4',
        # 'incl 1e-3', 
        # 'incl 1e-2', 
        # 'incl 0.95',
        # 'owa 1e-6',
        # 'quickreduct 1e-6',
        'cv-thres',
        # 'frfs'
    ],
    'quickreduct' : [
        'incl 1e-6',
        'quickreduct 1e-6',
        'frfs'
    ],
    'gran' : [
        'base-no-reducts',
        'msecvx-no-reducts',
        'maecvx-no-reducts'
    ]
}

In [253]:
nr = 'gran' # 6

In [254]:
main_folder = nr # + "small"  # 'tables_inclusion_set2' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [255]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [256]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]

In [257]:
# table_acc['max'] = table_acc.max(axis='columns', skipna=True)

In [258]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [259]:
table_acc

,base-no-reducts,msecvx-no-reducts,maecvx-no-reducts
australian,0.791167,0.837184,0.846763
bands,0.659821,0.527937,0.528872
bupa,0.563404,0.495208,0.494375
cleveland,0.26418,0.267832,0.290801
dermatology,0.849325,0.570859,0.612652
ecoli,0.708206,0.247619,0.247619
glass,0.529702,0.180496,0.223552
heart,0.740833,0.725,0.770833
ionosphere,0.945493,0.633974,0.651282
pima,0.630744,0.526771,0.5


In [260]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_142/3326315922.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [261]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [262]:
table_rule

,base-no-reducts,msecvx-no-reducts,maecvx-no-reducts
australian,176.2,17.7,17.9
bands,148.0,3.8,3.0
bupa,155.6,1.3,1.0
cleveland,165.5,35.3,36.7
dermatology,48.7,9.8,11.1
ecoli,86.8,2.9,2.9
glass,84.4,3.7,3.1
heart,105.2,30.4,30.5
ionosphere,69.2,5.7,4.8
pima,258.1,1.9,1.0


In [263]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_142/3026783303.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [264]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)

In [265]:
table_attribute

,base-no-reducts,msecvx-no-reducts,maecvx-no-reducts
australian,14.0,14.0,14.0
bands,19.0,19.0,19.0
bupa,6.0,6.0,6.0
cleveland,13.0,13.0,13.0
dermatology,34.0,34.0,34.0
ecoli,7.0,7.0,7.0
glass,9.0,9.0,9.0
heart,13.0,13.0,13.0
ionosphere,33.0,33.0,33.0
pima,8.0,8.0,8.0


In [231]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_142/3212901870.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [151]:
from scipy import stats
import scikit_posthocs as sph

In [203]:
subject = table_acc

In [204]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('max', axis='columns', inplace=True)

In [205]:
no_mean

,incl 1e-6,cv-thres
australian,0.824936,0.812028
bands,0.664654,0.658848
bupa,0.593125,0.589394
cleveland,0.317538,0.293392
dermatology,0.903319,0.903319
ecoli,0.700635,0.682324
glass,0.668254,0.668552
heart,0.730833,0.769167
ionosphere,0.910721,0.901234
pima,0.680919,0.657748


In [208]:
stats.wilcoxon(no_mean['cv-thres'],no_mean['incl 1e-6'], alternative='two-sided')

/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "


WilcoxonResult(statistic=19.0, pvalue=0.03546470751403722)

In [128]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=72.60000000000002, pvalue=1.5315170910874449e-09)

In [129]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
incl 1e-6,1.000000,1.0,1.0,0.757743,0.858925
incl 1e-4,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-3,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-2,0.757743,1.0,1.0,1.000000,1.000000
cv-thres,0.858925,1.0,1.0,1.000000,1.000000


In [65]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

incl 1e-3     0.04599
incl 1e-6    0.047052
cv-thres     0.050783
incl 1e-4    0.050873
incl 1e-2    0.181944
dtype: object

In [66]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
incl 1e-6 &  2.500000 \\
incl 1e-3 &  2.833333 \\
incl 1e-4 &  2.866667 \\
cv-thres  &  3.300000 \\
incl 1e-2 &  3.500000 \\
\bottomrule
\end{tabular}


/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_24131/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [67]:
ranks['lower-new-check'].value_counts()

KeyError: 'lower-new-check'

In [68]:
ranks.apply(pd.value_counts)

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
1.0,3,5.0,1.0,3,2
2.0,4,3.0,3.0,1,2
2.5,1,NaN,1.0,1,1
3.0,5,1.0,7.0,1,2
4.0,1,1.0,3.0,3,5
5.0,1,5.0,NaN,6,3


On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000
